## Load Data

In [ ]:
import pandas as pd
from tabulate import tabulate
from datetime import datetime

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_excel("dataset_umkm_01.xlsx")

print(df.head())

    id_umkm                 nama_usaha  jenis_usaha  tenaga_kerja_perempuan  \
0  28828567         Ud. Alif Pamungkas    Kesehatan                       1   
1  28828568          Ud. Zidanar Panji  Perdagangan                       5   
2  28828570         Ud. Damaris Satria         Jasa                      89   
3  28828571         Ud. Qasya Wiratama         Jasa                      91   
4  28828572  Ud. Grego Wiraatmaja Nara         Jasa                      76   

   tenaga_kerja_laki_laki       aset     omset      marketplace  \
0                      56  5497149.0   3347794        Tokopedia   
1                      44  7398384.0  39969661        Bukalapak   
2                       5  9576178.0  20700389  Website Sendiri   
3                      59  9456853.0   4820810           Lazada   
4                      36  9072119.0  19297316        Bukalapak   

   kapasitas_produksi  tahun_berdiri         laba  biaya_karyawan  \
0                 479           2015 -178079615.0    

## Data Preprocessing

Pada tahap preprocessing, dilakukan pembersihan data dari nilai ekstrem (data usaha rugi >50 juta), konversi satuan ke juta rupiah, pengelompokan aset menjadi kategori modal, serta penambahan variabel umur usaha untuk mendukung analisis potensi usaha.

In [ ]:
# =========================
# 2. PREPROCESSING
# =========================

# Hapus outlier kasar
df = df[df["laba"] > -50000000]

# Konversi ke juta
df["laba_juta"] = df["laba"] / 1000000
df["omset_juta"] = df["omset"] / 1000000

# Kategori modal
df["kategori_modal"] = pd.qcut(df["aset"], q=3, labels=["kecil", "sedang", "besar"])

# Umur usaha
current_year = datetime.now().year
df["umur_usaha"] = current_year - df["tahun_berdiri"]

## Range Modal

Sistem menghitung rentang nilai aset untuk setiap kategori modal dengan mengambil nilai minimum dan maksimum, kemudian menampilkannya dalam bentuk range untuk memudahkan interpretasi.

In [ ]:
# =========================
# 3. RANGE MODAL
# =========================
range_kategori = df.groupby("kategori_modal")["aset"].agg(["min", "max"]).reset_index()

def get_range_modal(kategori): # Mengambil dan menampilkan range modal sesuai kategori
    row = range_kategori[range_kategori["kategori_modal"] == kategori] # Mencari data sesuai kategori input
    if not row.empty:
        return f"{int(row['min'].values[0]):,} - {int(row['max'].values[0]):,}" # Jika kategori ada, maka mengambil nilai minimum dan maksimum. Kemudian di format
    return "-"

/tmp/ipykernel_4292/1207410673.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  range_kategori = df.groupby("kategori_modal")["aset"].agg(["min", "max"]).reset_index()


## Forward Chaining

Forward chaining terdapat pada fungsi forward_chaining, yaitu saat sistem memfilter data berdasarkan input pengguna berupa jenis usaha dan kategori modal. Proses ini merupakan implementasi rule-based reasoning dari fakta menuju kesimpulan.

In [ ]:
# =========================
# 4. FILTER DATA
# =========================
def forward_chaining(jenis_usaha, kategori_modal):
    return df[
        (df["jenis_usaha"].str.lower() == jenis_usaha.lower()) &
        (df["kategori_modal"] == kategori_modal)
    ]

## Scoring Potensi

Fungsi ini menghitung skor potensi usaha berdasarkan beberapa indikator seperti persentase keuntungan, jumlah pelanggan, efisiensi aset, dan umur usaha yang digabung menggunakan bobot tertentu.

In [ ]:
# =========================
# 5. SCORING POTENSI
# =========================
def hitung_skor(data):
    persen_untung = (len(data[data["laba"] > 0]) / len(data)) * 100
    rata_pelanggan = data["jumlah_pelanggan"].mean()
    rata_roa = data["roa"].mean()
    rata_umur = data["umur_usaha"].mean()

    # Normalisasi sederhana (biar skala seimbang)
    skor = (
        (persen_untung * 0.4) +
        (rata_pelanggan * 0.2) +
        (rata_roa * 0.2) +
        (rata_umur * 0.2)
    )
    return round(skor, 2), persen_untung

## Rekomendasi

Fungsi ini digunakan untuk menghasilkan rekomendasi usaha dengan cara mengambil data sesuai input, menghitung indikator penting, menentukan skor potensi, lalu menghasilkan status dan penjelasan dalam bentuk tabel.

In [ ]:
# =========================
# 6. REKOMENDASI
# =========================
def get_rekomendasi(jenis_usaha, kategori_modal):

    data = forward_chaining(jenis_usaha, kategori_modal)

    if data.empty:
        return pd.DataFrame({
            "Status": ["Tidak ada data"],
            "Penjelasan": ["Data tidak ditemukan"]
        })

    # Statistik
    median_omset = round(data["omset_juta"].median(), 2)
    median_laba = round(data["laba_juta"].median(), 2)

    top_daerah = data["daerah"].value_counts().idxmax()
    top3_daerah = ", ".join(data["daerah"].value_counts().head(3).index)

    # Skor
    skor, persen_untung = hitung_skor(data)

    # Status
    if skor >= 60:
        status = "Potensial"
    elif skor >= 40:
        status = "Perlu Strategi"
    else:
        status = "Berisiko Tinggi"

    # Insight
    penjelasan = (
        f"Usaha {jenis_usaha} dengan modal {kategori_modal} "
        f"memiliki tingkat keberhasilan {persen_untung:.1f}%. "
        f"Daerah paling potensial adalah {top_daerah}. "
        f"Median omset Rp {median_omset} juta dan laba Rp {median_laba} juta. "
        f"Skor potensi usaha sebesar {skor}. "
        f"Disarankan memilih lokasi dengan permintaan tinggi dan mengelola modal secara efisien."
    )

    return pd.DataFrame({
        "Jenis Usaha": [jenis_usaha],
        "Kategori Modal": [kategori_modal],
        "Range Modal": [get_range_modal(kategori_modal)],
        "Top Daerah": [top_daerah],
        "Daerah Potensial": [top3_daerah],
        "Median Omset (Juta)": [median_omset],
        "Median Laba (Juta)": [median_laba],
        "Persentase Untung (%)": [round(persen_untung, 2)],
        "Skor": [skor],
        "Status": [status],
        "Penjelasan": [penjelasan]
    })

def tampilkan_tabel(df_hasil):
    print("\n=== HASIL REKOMENDASI USAHA ===\n")

    # Batasi panjang kolom penjelasan
    df_copy = df_hasil.copy()
    df_copy["Penjelasan"] = df_copy["Penjelasan"].str.slice(0, 120) + "..."

    print(tabulate(df_copy, headers='keys', tablefmt='grid', showindex=False))

## Contoh Hasil Rekomendasi

In [ ]:
# =========================
# 7. CONTOH RUN
# =========================
hasil = get_rekomendasi("Jasa", "kecil")
tampilkan_tabel(hasil)


=== HASIL REKOMENDASI USAHA ===

+---------------+------------------+---------------------+--------------+----------------------+-----------------------+----------------------+-------------------------+--------+-----------+-----------------------------------------------------------------------------------------------------------------------------+
| Jenis Usaha   | Kategori Modal   | Range Modal         | Top Daerah   | Daerah Potensial     |   Median Omset (Juta) |   Median Laba (Juta) |   Persentase Untung (%) |   Skor | Status    | Penjelasan                                                                                                                  |
+===============+==================+=====================+==============+======================+=======================+======================+=========================+========+===========+=============================================================================================================================+
| Jasa         

“Usaha tersebut dikatakan potensial karena berdasarkan data historis memiliki persentase keuntungan yang tinggi, jumlah pelanggan yang besar, serta efisiensi penggunaan aset yang baik. Selain itu, usaha-usaha sejenis juga terbukti mampu bertahan dalam jangka waktu yang cukup lama.”

**KESIMPULAN**

Kode ini melakukan:

1. Membersihkan data

2. Mengelompokkan modal

3. Memfilter berdasarkan input user

4. Menghitung potensi usaha (scoring)

5. Menentukan status (potensial / risiko)

6. Menampilkan hasil dalam tabel + penjelasan